In [1]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

In [2]:
np.random.seed(42)

damage_classes = [
    "Front Breakage",
    "Front Crushed",
    "Rear Breakage",
    "Rear Crushed"
]

# Base repair cost for each damage type
base_cost = {
    "Front Breakage": 40000,
    "Front Crushed": 90000,
    "Rear Breakage": 35000,
    "Rear Crushed": 80000
}

n_samples = 3000
rows = []

for _ in range(n_samples):
    damage = np.random.choice(damage_classes)

    # Vehicle information
    vehicle_age = np.random.randint(0, 15)       # years
    mileage = np.random.randint(0, 200000)       # kilometers

    # Synthetic repair cost formula
    cost = (
        base_cost[damage]
        + vehicle_age * 1200
        + mileage * 0.05
        + np.random.normal(0, 5000)
    )

    # Ensure minimum repair cost
    cost = max(1000, cost)

    rows.append([
        damage,
        vehicle_age,
        mileage,
        round(cost, 2)
    ])

df = pd.DataFrame(
    rows,
    columns=[
        "damage_class",
        "vehicle_age",
        "mileage",
        "estimated_cost"
    ]
)

df.head()

,damage_class,vehicle_age,mileage,estimated_cost
0,Rear Breakage,3,131932,47915.32
1,Front Breakage,6,137337,50989.70
2,Rear Breakage,6,168266,40558.49
3,Rear Crushed,7,41090,87990.48
4,Front Crushed,4,769,96003.93


In [3]:
os.makedirs("../data/synthetic", exist_ok=True)

df.to_csv(
    "../data/synthetic/repair_cost_data.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.


In [4]:
print(df.shape)
print(df.describe())
print(df["damage_class"].value_counts())

(3000, 4)
       vehicle_age        mileage  estimated_cost
count  3000.000000    3000.000000     3000.000000
mean      7.039333   99055.518333    74713.188327
std       4.263001   57583.916103    25146.741427
min       0.000000      60.000000    29107.070000
25%       3.000000   49082.250000    50791.512500
50%       7.000000   98833.500000    76981.080000
75%      11.000000  148770.250000    98441.425000
max      14.000000  199988.000000   125543.800000
damage_class
Rear Crushed      762
Rear Breakage     754
Front Crushed     748
Front Breakage    736
Name: count, dtype: int64


In [5]:
X = df[["damage_class", "vehicle_age", "mileage"]]
y = df["estimated_cost"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [7]:
categorical_features = ["damage_class"]
numeric_features = ["vehicle_age", "mileage"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)

In [8]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            XGBRegressor(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42
            )
        )
    ]
)

In [9]:
model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


In [10]:
y_pred = model.predict(X_test)

In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:.4f}")

MAE  : 4,065.76
RMSE : 5,085.07
R²   : 0.9588


In [13]:
os.makedirs("../saved_models", exist_ok=True)

joblib.dump(
    model,
    "../saved_models/repair_cost_model.pkl"
)

print("Model saved successfully.")

Model saved successfully.


In [14]:
sample = pd.DataFrame({
    "damage_class": ["Front Breakage"],
    "vehicle_age": [4],
    "mileage": [55000]
})

predicted_cost = model.predict(sample)[0]

print(f"Estimated Repair Cost: ₹{predicted_cost:,.2f}")

Estimated Repair Cost: ₹45,325.30


In [15]:
loaded_model = joblib.load(
    "../saved_models/repair_cost_model.pkl"
)

loaded_prediction = loaded_model.predict(sample)[0]

print(f"Loaded Model Prediction: ₹{loaded_prediction:,.2f}")

Loaded Model Prediction: ₹45,325.30
